In [1]:
# DETECCION DE ANUNCIOS ANOMALOS
#
# Objetivo: senalar anuncios que se salen de lo normal, ya sea por
# error de publicacion, por corresponder a un producto distinto al
# que dicen, o por posible intento de fraude.
#
# IMPORTANTE: el sistema senala INDICIOS ESTADISTICOS. Que un
# anuncio aparezca marcado no significa que sea fraudulento, solo
# que se comporta de forma distinta al resto y merece revision.
#
# Se combinan cuatro senales independientes:
#   1. Reglas de consistencia (valores imposibles)
#   2. Desviacion respecto a su barrio
#   3. Desviacion respecto a lo que predice el modelo de precio
#   4. Deteccion automatica de combinaciones raras

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import IsolationForest

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

df = pd.read_parquet("../data/processed/model_dataset.parquet")
modelo = joblib.load("../models/artifacts/price_model.pkl")

NUMERICAS = ["area_sqm", "n_rooms", "sqm_per_room"]
CATEGORICAS = ["typology_grouped", "zona_modelo", "river_bank"]

print(f"{len(df)} anuncios a revisar")
print(f"(se incluyen los marcados en la fase 2: {df['is_outlier'].sum()})")

859 anuncios a revisar
(se incluyen los marcados en la fase 2: 3)


In [2]:
# NIVEL 1: REGLAS DE CONSISTENCIA
#
# Valores que no pueden ser correctos. Los umbrales se fijaron en
# el diccionario de datos a partir de la distribucion observada,
# antes de construir ningun modelo.

df["regla_precio_m2"] = (
    (df["price_per_sqm"] < 5) | (df["price_per_sqm"] > 28)
).fillna(False)

df["regla_superficie"] = (
    (df["area_sqm"] < 25) | (df["area_sqm"] > 300)
).fillna(False)

df["regla_habitaciones"] = (df["n_rooms"] > 8).fillna(False)

# Un piso de 30 m2 con 4 habitaciones no es creible
df["regla_densidad"] = (df["sqm_per_room"] < 12).fillna(False)

reglas = ["regla_precio_m2", "regla_superficie",
          "regla_habitaciones", "regla_densidad"]

df["n_reglas"] = df[reglas].sum(axis=1)

print("REGLAS ACTIVADAS")
for r in reglas:
    print(f"  {r:22s} {df[r].sum():3d} anuncios")
print()
print(f"Anuncios con alguna regla: {(df['n_reglas'] > 0).sum()}")

REGLAS ACTIVADAS
  regla_precio_m2          3 anuncios
  regla_superficie         0 anuncios
  regla_habitaciones       0 anuncios
  regla_densidad           0 anuncios

Anuncios con alguna regla: 3


In [3]:
# NIVEL 2: CUANTO SE DESVIA DE SU BARRIO
#
# price_relative_zone ya compara cada piso con la mediana de su
# zona, EXCLUYENDO el propio anuncio del calculo.
#
# Se convierte en z-score: cuantas desviaciones tipicas se aleja
# de lo normal. Un valor absoluto por encima de 3 es raro.

media = df["price_relative_zone"].mean()
desv = df["price_relative_zone"].std()
df["z_zona"] = ((df["price_relative_zone"] - media) / desv).round(2)

df["senal_zona"] = df["z_zona"].abs() > 2.5

print(f"Media de price_relative_zone: {media:.3f}")
print(f"Desviacion tipica: {desv:.3f}")
print()
print(f"Anuncios con |z| > 2,5: {df['senal_zona'].sum()}")
print()
print("Los mas desviados:")
cols = ["zona_modelo", "monthly_price", "area_sqm", "price_per_sqm", "z_zona"]
print(df.nlargest(5, "z_zona")[cols].round(2).to_string(index=False))
print()
print(df.nsmallest(5, "z_zona")[cols].round(2).to_string(index=False))

Media de price_relative_zone: 1.044
Desviacion tipica: 0.272

Anuncios con |z| > 2,5: 19

Los mas desviados:
       zona_modelo  monthly_price  area_sqm  price_per_sqm  z_zona
 Plentzia|Plentzia         6000.0     180.0          33.33    9.11
 Plentzia|Plentzia         2500.0      85.0          29.41    7.59
 Plentzia|Plentzia         5000.0     212.0          23.58    5.32
Bilbao|Casco Viejo         1875.0      70.0          26.79    3.96
      Getxo|Neguri         1100.0      40.0          27.50    3.89

       zona_modelo  monthly_price  area_sqm  price_per_sqm  z_zona
     Sopela|Sopela          550.0     160.0           3.44   -2.87
Bilbao|Casco Viejo         1500.0     256.0           5.86   -2.15
     Bilbao|Abando         1400.0     170.0           8.24   -1.82
   Bilbao|Indautxu         1450.0     180.0           8.06   -1.72
    margen|derecha         1600.0     260.0           6.15   -1.71


In [4]:
# NIVEL 3: CUANTO SE DESVIA DE LO QUE PREDICE EL MODELO
#
# Esta es la senal mas potente. El modelo estima lo que deberia
# costar un piso con esas caracteristicas en ese barrio. Si el
# anuncio pide mucho mas o mucho menos, algo no encaja.
#
# Solo se puede calcular para los anuncios con superficie: el
# modelo la necesita.

con_datos = df[df["area_sqm"].notna()].copy()
pred_log = modelo.predict(con_datos[NUMERICAS + CATEGORICAS])
con_datos["precio_esperado"] = np.exp(pred_log).round(0)

# El residuo se calcula en logaritmo para que sea simetrico:
# pagar el doble y pagar la mitad deben pesar igual
con_datos["residuo"] = np.log(con_datos["monthly_price"]) - pred_log
r_std = con_datos["residuo"].std()
con_datos["z_modelo"] = (con_datos["residuo"] / r_std).round(2)

df["precio_esperado"] = con_datos["precio_esperado"]
df["z_modelo"] = con_datos["z_modelo"]
df["senal_modelo"] = df["z_modelo"].abs() > 2.5

print(f"Desviacion tipica del residuo: {r_std:.3f}")
print(f"Anuncios con |z| > 2,5: {df['senal_modelo'].sum()}")
print()
print("MAS CAROS DE LO ESPERADO")
cols = ["zona_modelo", "area_sqm", "n_rooms", "monthly_price",
        "precio_esperado", "z_modelo"]
print(df.nlargest(8, "z_modelo")[cols].to_string(index=False))
print()
print("MAS BARATOS DE LO ESPERADO")
print(df.nsmallest(8, "z_modelo")[cols].to_string(index=False))

Desviacion tipica del residuo: 0.147
Anuncios con |z| > 2,5: 15

MAS CAROS DE LO ESPERADO
       zona_modelo  area_sqm  n_rooms  monthly_price  precio_esperado  z_modelo
 Plentzia|Plentzia     180.0        5         6000.0           2099.0      7.16
 Plentzia|Plentzia      85.0        3         2500.0            933.0      6.72
    margen|derecha      80.0        3         2100.0           1024.0      4.90
Bilbao|Casco Viejo      70.0        1         1875.0            961.0      4.55
 Plentzia|Plentzia     100.0        2         1800.0           1127.0      3.19
     Bilbao|Bilbao     105.0        3         2300.0           1459.0      3.10
     Bilbao|Abando      95.0        2         2200.0           1412.0      3.02
Bilbao|Casco Viejo      65.0        1         1500.0            964.0      3.02

MAS BARATOS DE LO ESPERADO
         zona_modelo  area_sqm  n_rooms  monthly_price  precio_esperado  z_modelo
       Sopela|Sopela     160.0        1          550.0           1473.0     -6.7

In [5]:
# NIVEL 4: DETECCION AUTOMATICA DE COMBINACIONES RARAS
#
# Isolation Forest no busca nada concreto: aisla los puntos que
# resultan faciles de separar del resto. La idea es que un caso
# raro necesita pocas divisiones para quedar solo.
#
# Encuentra combinaciones extranas que las reglas no preveen:
# por ejemplo un piso grande, barato y con muchas habitaciones
# en una zona cara.

vars_iso = ["monthly_price", "area_sqm", "n_rooms",
            "price_per_sqm", "sqm_per_room"]

datos_iso = df[vars_iso].dropna()

iso = IsolationForest(
    contamination=0.05,        # se espera un 5 % de anomalos
    n_estimators=200,
    random_state=RANDOM_STATE,
)
iso.fit(datos_iso)

# score negativo = mas anomalo. Se invierte para que mas alto
# signifique mas raro
puntuacion = -iso.score_samples(datos_iso)
df.loc[datos_iso.index, "score_iso"] = puntuacion.round(3)
df["senal_iso"] = iso.predict(datos_iso) == -1 if False else False
df.loc[datos_iso.index, "senal_iso"] = (iso.predict(datos_iso) == -1)

print(f"Anuncios evaluados: {len(datos_iso)}")
print(f"Marcados como anomalos: {df['senal_iso'].sum()}")
print()
print("LOS 8 MAS ANOMALOS SEGUN EL ALGORITMO")
cols = ["zona_modelo", "monthly_price", "area_sqm", "n_rooms",
        "price_per_sqm", "score_iso"]
print(df.nlargest(8, "score_iso")[cols].to_string(index=False))

Anuncios evaluados: 859
Marcados como anomalos: 43

LOS 8 MAS ANOMALOS SEGUN EL ALGORITMO
       zona_modelo  monthly_price  area_sqm  n_rooms  price_per_sqm  score_iso
 Plentzia|Plentzia         6000.0     180.0        5          33.33      0.748
 Plentzia|Plentzia         5000.0     212.0        3          23.58      0.742
Bilbao|Casco Viejo         1500.0     256.0        2           5.86      0.735
     Sopela|Sopela          550.0     160.0        1           3.44      0.731
       Getxo|otros         2600.0     250.0        6          10.40      0.695
  margen|txorierri         1800.0     280.0        5           6.43      0.690
     Bilbao|Abando         3500.0     220.0        4          15.91      0.677
    margen|derecha         1600.0     260.0        4           6.15      0.670


In [6]:
# PUNTUACION FINAL DE ANOMALIA
#
# Se combinan las cuatro senales en un valor entre 0 y 1.
#
# Los pesos reflejan la confianza en cada senal:
#   - Las reglas son las mas fiables: un valor imposible es un error
#   - La desviacion del modelo es muy informativa porque considera
#     todas las caracteristicas a la vez
#   - La desviacion de zona es util pero mas simple
#   - Isolation Forest aporta casos que los demas no ven, pero
#     tambien mas falsos positivos

PESOS = {"reglas": 0.35, "modelo": 0.30, "zona": 0.20, "iso": 0.15}

def normaliza(serie, tope=4.0):
    """Lleva un z-score a un valor entre 0 y 1, con tope."""
    return (serie.abs().clip(0, tope) / tope).fillna(0)

df["p_reglas"] = (df["n_reglas"] / len(reglas)).clip(0, 1)
df["p_zona"]   = normaliza(df["z_zona"])
df["p_modelo"] = normaliza(df["z_modelo"])
df["p_iso"]    = ((df["score_iso"] - df["score_iso"].min()) /
                  (df["score_iso"].max() - df["score_iso"].min())).fillna(0)

df["anomaly_score"] = (
    PESOS["reglas"] * df["p_reglas"] +
    PESOS["modelo"] * df["p_modelo"] +
    PESOS["zona"]   * df["p_zona"] +
    PESOS["iso"]    * df["p_iso"]
).round(4)

df["n_senales"] = (df[["senal_zona", "senal_modelo", "senal_iso"]].sum(axis=1)
                   + (df["n_reglas"] > 0).astype(int))

print("DISTRIBUCION DE LA PUNTUACION")
print(df["anomaly_score"].describe().round(3))
print()
print("ANUNCIOS POR NUMERO DE SENALES ACTIVADAS")
print(df["n_senales"].value_counts().sort_index())
print()
print("LOS 15 MAS ANOMALOS")
cols = ["zona_modelo", "monthly_price", "area_sqm", "n_rooms",
        "price_per_sqm", "precio_esperado", "n_senales", "anomaly_score"]
print(df.nlargest(15, "anomaly_score")[cols].to_string(index=False))

DISTRIBUCION DE LA PUNTUACION
count    859.000
mean       0.112
std        0.085
min        0.008
25%        0.056
50%        0.092
75%        0.147
max        0.738
Name: anomaly_score, dtype: float64

ANUNCIOS POR NUMERO DE SENALES ACTIVADAS
n_senales
0    804
1     38
2     12
3      2
4      3
Name: count, dtype: int64

LOS 15 MAS ANOMALOS
       zona_modelo  monthly_price  area_sqm  n_rooms  price_per_sqm  precio_esperado  n_senales  anomaly_score
 Plentzia|Plentzia         6000.0     180.0        5          33.33           2099.0          4         0.7375
 Plentzia|Plentzia         2500.0      85.0        3          29.41            933.0          4         0.7001
     Sopela|Sopela          550.0     160.0        1           3.44           1473.0          4         0.6742
Bilbao|Casco Viejo         1875.0      70.0        1          26.79            961.0          3         0.6082
    margen|derecha         2100.0      80.0        3          26.25           1024.0          2    

In [9]:
# REVISION MANUAL PARA EVALUAR EL SISTEMA
#
# Al no existir etiquetas de fraude, la unica forma de medir si el
# detector funciona es revisar los casos que senala.
#
# Se exportan los 50 con mayor puntuacion, con dos columnas vacias
# para completar a mano durante la revision.

cols = ["listing_id", "zona_modelo", "typology",
        "monthly_price", "area_sqm", "n_rooms",
        "price_per_sqm", "precio_esperado",
        "z_zona", "z_modelo", "n_senales", "anomaly_score"]

top50 = df.nlargest(50, "anomaly_score")[cols].round(2).copy()

# Columnas a rellenar durante la revision
top50["es_anomalo"] = ""
top50["motivo"] = ""

ruta = "../data/processed/revision_anomalias.csv"
top50.to_csv(ruta, index=False, sep=";", encoding="utf-8-sig")

print(f"Exportados {len(top50)} casos a {ruta}")
print("Separador: punto y coma | Codificacion: UTF-8 con BOM")
print()
print(top50.head(10).to_string(index=False))

Exportados 50 casos a ../data/processed/revision_anomalias.csv
Separador: punto y coma | Codificacion: UTF-8 con BOM

 listing_id        zona_modelo  typology  monthly_price  area_sqm  n_rooms  price_per_sqm  precio_esperado  z_zona  z_modelo  n_senales  anomaly_score es_anomalo motivo
        638  Plentzia|Plentzia     house         6000.0     180.0        5          33.33           2099.0    9.11      7.16          4           0.74                  
        767  Plentzia|Plentzia      flat         2500.0      85.0        3          29.41            933.0    7.59      6.72          4           0.70                  
        865      Sopela|Sopela      flat          550.0     160.0        1           3.44           1473.0   -2.87     -6.71          4           0.67                  
        576 Bilbao|Casco Viejo      flat         1875.0      70.0        1          26.79            961.0    3.96      4.55          3           0.61                  
        505     margen|derecha      f

In [14]:
# EVALUACION DEL DETECTOR
#
# Al no existir etiquetas de fraude, la unica forma de medir el
# sistema es revisar manualmente los casos que senala.
#
# Se reporta la precision por tramos y no solo el valor global,
# porque un detector puede ser muy preciso en los casos mas
# extremos y perder acierto conforme desciende la puntuacion.
# Esa informacion es la que permite fijar un umbral operativo.

# Excel guarda en la codificacion de Windows, no en UTF-8.
# Se prueban las habituales hasta dar con la correcta.
for cod in ["utf-8-sig", "cp1252", "latin-1"]:
    try:
        rev = pd.read_csv("../data/processed/revision_anomalias.csv",
                          sep=";", encoding=cod)
        print(f"Leido con codificacion: {cod}")
        break
    except UnicodeDecodeError:
        continue

rev["es_anomalo"] = rev["es_anomalo"].astype(str).str.strip().str.lower()
rev["motivo"] = rev["motivo"].astype(str).str.strip().str.lower()

# La categoria de mercado tenso NO constituye anomalia: son precios
# elevados que el modelo predice correctamente. Se recodifican para
# que la metrica refleje unicamente la deteccion de casos que el
# sistema no logra explicar.
rev.loc[rev["motivo"] == "mercado_tenso", "es_anomalo"] = "no"

rev = rev.sort_values("anomaly_score", ascending=False).reset_index(drop=True)

print()
print("RESULTADO DE LA REVISION")
print(rev["motivo"].value_counts().to_string())
print()
print(f"Anomalias reales: {(rev['es_anomalo'] == 'si').sum()} de {len(rev)}")

print()
print("PRECISION POR TRAMOS")
for k in [10, 20, 25, 50]:
    tramo = rev.head(k)
    aciertos = (tramo["es_anomalo"] == "si").sum()
    print(f"  Precision@{k:<3d} {aciertos:2d}/{k:2d}  =  {aciertos/k*100:5.1f} %")

print()
print("POSICION DE CADA TIPO DE HALLAZGO")
print("(en que puesto del ranking aparece el primero de cada clase)")
for m, n in rev["motivo"].value_counts().items():
    pos = rev.index[rev["motivo"] == m].min() + 1
    es_anom = "anomalia" if m in ("temporada", "precio_alto",
                                  "precio_bajo", "duplicado") else "no anomalia"
    print(f"  {m:16s} {n:2d} casos | primero en posicion {pos:2d} | {es_anom}")

Leido con codificacion: cp1252

RESULTADO DE LA REVISION
motivo
mercado_tenso    21
precio_alto      10
normal            7
temporada         5
duplicado         5
precio_bajo       2

Anomalias reales: 22 de 50

PRECISION POR TRAMOS
  Precision@10  10/10  =  100.0 %
  Precision@20  18/20  =   90.0 %
  Precision@25  19/25  =   76.0 %
  Precision@50  22/50  =   44.0 %

POSICION DE CADA TIPO DE HALLAZGO
(en que puesto del ranking aparece el primero de cada clase)
  mercado_tenso    21 casos | primero en posicion 14 | no anomalia
  precio_alto      10 casos | primero en posicion  4 | anomalia
  normal            7 casos | primero en posicion 21 | no anomalia
  temporada         5 casos | primero en posicion  1 | anomalia
  duplicado         5 casos | primero en posicion 18 | anomalia
  precio_bajo       2 casos | primero en posicion  3 | anomalia


In [15]:
# LOS CASOS DE MERCADO TENSO
#
# Durante la revision se identifico una categoria intermedia:
# anuncios con precios elevados que el modelo predice
# correctamente. No constituyen anomalias, sino manifestacion de
# la tension del mercado.
#
# Su cuantificacion aporta un resultado complementario al objetivo
# original del componente.

tenso = rev[rev["motivo"] == "mercado_tenso"]
print(f"Anuncios en la categoria de mercado tenso: {len(tenso)}")
print()
print(f"  Precio medio:        {tenso['monthly_price'].mean():.0f} EUR")
print(f"  Precio por m2 medio: {tenso['price_per_sqm'].mean():.1f} EUR")
print(f"  Superficie media:    {tenso['area_sqm'].mean():.0f} m2")
print()
print("Distribucion por zona:")
print(tenso["zona_modelo"].value_counts().head(8).to_string())

Anuncios en la categoria de mercado tenso: 21

  Precio medio:        1735 EUR
  Precio por m2 medio: 18.9 EUR
  Superficie media:    94 m2

Distribucion por zona:
zona_modelo
Bilbao|Abando                     5
Getxo|Neguri                      3
Getxo|otros                       2
Bilbao|Bilbao                     2
Bilbao|Campo Volant¡n-Casta¤os    1
Sopela|Sopela                     1
Bilbao|Indautxu                   1
Barakaldo|otros                   1
